The [TensorFlow Embedding Projector](https://projector.tensorflow.org/) places
high-dimensional word vectors in a three-dimensional map where distance approximates
semantic similarity, and lets you pick a word to see its nearest neighbors. In this
assignment you build the same thing in PyTorch: you train word embeddings with
`torch.nn.Embedding`, project them to three dimensions, draw an interactive scatter, and
query the neighborhood of any token.

You will complete the parts marked with `TODO(you)`. Each raises `NotImplementedError`
until you implement it.

In [1]:
import re
from collections import Counter
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

## Corpus and vocabulary

Word embeddings are learned from co-occurrence in text. Load a compact corpus, keep the
most frequent words as the vocabulary, and turn the text into a stream of integer ids.

In [2]:
from datasets import load_dataset

raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
text = " ".join(raw["text"]).lower()
tokens = re.findall(r"[a-z]+", text)[:300_000]
counts = Counter(tokens)

V = 8000
vocab = [word for word, count in counts.most_common(V)]

word2idx = {word: i for i, word in enumerate(vocab)}

idx2word = {i: word for i, word in enumerate(vocab)}

corpus = [word2idx[word] for word in tokens if word in word2idx]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/validation-00000-of-00(…): reconstructing file:   0%|          |  0.00B /  657kB            

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

## Word2vec embeddings with softmax and cross-entropy

A word2vec model learns word embeddings by predicting context words. This is the skip-gram
architecture of word2vec: the center word predicts its context. The center word's embedding is
scored against every word in the vocabulary, a softmax turns those scores into a probability
distribution over possible context words, and the cross-entropy loss pushes up the probability
of the true context word:

$$p(o \mid c) = \frac{\exp(\mathbf{c}\cdot\mathbf{v}_o)}{\sum_{w}\exp(\mathbf{c}\cdot\mathbf{v}_w)},
\qquad L = -\log p(o \mid c).$$

The learned center embedding table is the word-vector matrix you will project.

In [3]:
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.center = nn.Embedding(vocab_size, dim)   # word vectors
        self.output = nn.Linear(dim, vocab_size)      # score every word as a possible context
        nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)

    def forward(self, center_ids):
        x = self.center(center_ids)
        logits = self.output(x)
        return logits

In [4]:
# Build (center, context) pairs from a sliding window
window = 3
pairs = []
for i, wc in enumerate(corpus):
    for j in range(max(0, i - window), min(len(corpus), i + window + 1)):
        if j != i:
            pairs.append((wc, corpus[j]))
pairs = np.array(pairs, dtype=np.int64)

dim, B, epochs = 64, 1024, 3
model = Word2Vec(V, dim)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(epochs):
    np.random.shuffle(pairs)
    total_loss = 0

    for i in range(0, len(pairs), B):
        batch = pairs[i:i+B]

        center_ids = torch.tensor(batch[:, 0], dtype=torch.long)
        context_ids = torch.tensor(batch[:, 1], dtype=torch.long)

        opt.zero_grad()

        logits = model(center_ids)
        loss = loss_fn(logits, context_ids)

        loss.backward()
        opt.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss:.4f}")

emb = model.center.weight.detach().cpu().numpy()

Epoch 1/3, Loss: 11303.1983
Epoch 2/3, Loss: 10861.1915
Epoch 3/3, Loss: 10638.5498


## Projecting the embeddings to three dimensions

The embedding matrix lives in $d=64$ dimensions. To see it, project a few thousand of the most
frequent words down to three dimensions. Principal component analysis is linear and fast; UMAP is
nonlinear and tends to separate clusters more sharply. The interactive scatter lets you rotate the
cloud and hover to read each word.

In [6]:
from sklearn.decomposition import PCA

N = 1500
plot_words = vocab[:N]
X = emb[:N]

pca3 = PCA(n_components=3).fit_transform(X)

try:
    import umap
    umap3 = umap.UMAP(n_components=3, random_state=42).fit_transform(X)
except ImportError:
    umap3 = None

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [7]:
import plotly.graph_objects as go

def plot_embeddings(coords, words, query=None, neighbor_set=None):
    neighbor_set = set() if neighbor_set is None else set(neighbor_set)

    colors = []
    sizes = []

    for word in words:
        if word == query:
            colors.append("red")
            sizes.append(10)
        elif word in neighbor_set:
            colors.append("orange")
            sizes.append(7)
        else:
            colors.append("blue")
            sizes.append(3)

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=coords[:, 0],
                y=coords[:, 1],
                z=coords[:, 2],
                mode="markers",
                text=words,
                hovertemplate="%{text}<extra></extra>",
                marker=dict(
                    size=sizes,
                    color=colors,
                    opacity=0.7
                )
            )
        ]
    )

    fig.update_layout(
        scene=dict(
            xaxis_title="Dimension 1",
            yaxis_title="Dimension 2",
            zaxis_title="Dimension 3"
        )
    )

    return fig

plot_embeddings(pca3, plot_words)

## Querying a token's neighborhood

The projector's key feature is the neighborhood query: pick a word and see its closest
words. Closeness is measured by cosine similarity in the full embedding space (not in the
3D projection). The query below returns the top-k neighbors and highlights them in the
scatter.

In [8]:
def neighbors(word, k=10):
    idx = word2idx[word]

    query_vec = emb[idx]

    query_norm = query_vec / np.linalg.norm(query_vec)
    emb_norm = emb / np.linalg.norm(emb, axis=1, keepdims=True)

    scores = emb_norm @ query_norm

    scores[idx] = -np.inf

    top_indices = np.argsort(scores)[::-1][:k]

    return [(idx2word[i], scores[i]) for i in top_indices]
for w, s in neighbors("government", 10):
    print(f"{w:15s} {s:.3f}")

federal         0.815
troops          0.814
municipal       0.797
authorities     0.793
pakistani       0.792
extending       0.789
subcontinent    0.788
courts          0.787
commonwealth    0.787
revolutionary   0.786


In [12]:
query = "government"

neighbor_list = neighbors(query, 10)
neighbor_set = {word for word, score in neighbor_list}

plot_embeddings(
    pca3,
    plot_words,
    query=query,
    neighbor_set=neighbor_set
)

## Exploration

Answer in the cells you add below.

1. Query several words of your choice (a few nouns, a verb, a function word). Which return clean
   semantic neighbors and which do not? Why might rare words give noisier neighbors?
2. Plot the clusters. Draw the projected embeddings (the UMAP layout separates clusters most
   clearly) and describe the groupings you see: do related words land near each other? Name a few
   clusters you can identify.

In [10]:
test_words = ["government", "war", "music", "run", "the"]

for word in test_words:
    print(f"\n{word.upper()}")
    for w, s in neighbors(word, 10):
        print(f"{w:15s} {s:.3f}")


GOVERNMENT
federal         0.815
troops          0.814
municipal       0.797
authorities     0.793
pakistani       0.792
extending       0.789
subcontinent    0.788
courts          0.787
commonwealth    0.787
revolutionary   0.786

WAR
world           0.755
transylvania    0.747
outbreak        0.741
romania         0.735
z               0.710
anglo           0.708
invaded         0.706
independence    0.701
syrian          0.691
dreadnoughts    0.689

MUSIC
accompanying    0.868
concept         0.795
pop             0.780
susan           0.767
video           0.761
videos          0.749
best            0.744
sony            0.741
producer        0.736
production      0.722

RUN
cheltenham      0.821
drew            0.775
fleetwood       0.773
equalised       0.771
struggling      0.768
defeat          0.752
buffalo         0.739
episodes        0.738
signing         0.737
away            0.736

THE
cuautla         0.739
gallia          0.738
tenth           0.737
opium           0.73

1. The nouns generally produced the clearest semantic neighbors. For example, government returned related words such as federal, municipal, authorities, and courts. War also produced relevant words such as outbreak, invaded, and independence. Music had some good neighbors such as pop, video, producer, and production, although it also included less directly related words. The verb run produced much noisier results, and the function word the produced almost entirely unrelated neighbors. Rare words can also have noisier neighbors because they appear fewer times in the training data, giving the model fewer examples from which to learn a reliable embedding.

In [11]:
plot_embeddings(umap3, plot_words)

2. The UMAP projection shows several noticeable groupings where words with similar meanings or contexts appear near one another. For example, I observed military and war-related words such as army, troops, battle, and navy; music and entertainment words such as music, album, song, recording, and producer; and religious words such as church, god, deity, and priest. There are also geographic and national terms grouped in similar regions. The clusters are not perfectly separated, and some unrelated words overlap, but overall the projection shows that Word2Vec learned meaningful relationships between words based on the contexts in which they appeared.